In [ ]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 16.1 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which 

In [ ]:
from datasets import load_dataset
raw_dataset=load_dataset('conll2003')

README.md:   0%|          | 0.00/12.3k [00:00<?, ?B/s]

conll2003.py:   0%|          | 0.00/9.57k [00:00<?, ?B/s]

The repository for conll2003 contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/conll2003.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

In [ ]:
raw_dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

In [ ]:
### rename , remove
raw_dataset=raw_dataset.remove_columns(['id','pos_tags','ner_tags'])
raw_dataset=raw_dataset.rename_column('tokens','words')
raw_dataset=raw_dataset.rename_column('chunk_tags','labels')
raw_dataset

DatasetDict({
    train: Dataset({
        features: ['words', 'labels'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['words', 'labels'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['words', 'labels'],
        num_rows: 3453
    })
})

In [ ]:
chunk_tags=raw_dataset['train'].features['labels']

In [ ]:
label_names=chunk_tags.feature.names
label_names

['O',
 'B-ADJP',
 'I-ADJP',
 'B-ADVP',
 'I-ADVP',
 'B-CONJP',
 'I-CONJP',
 'B-INTJ',
 'I-INTJ',
 'B-LST',
 'I-LST',
 'B-NP',
 'I-NP',
 'B-PP',
 'I-PP',
 'B-PRT',
 'I-PRT',
 'B-SBAR',
 'I-SBAR',
 'B-UCP',
 'I-UCP',
 'B-VP',
 'I-VP']

In [ ]:
words=raw_dataset['train']['words'][0]
labels=raw_dataset['train']['labels'][0]
line1=''
line2=''
for word,label in zip(words,labels):
  full_label=label_names[label]
  max_lenght=max(len(word),len(full_label))
  line1 += word + " "*(max_lenght-len(word)+1)
  line2 += full_label + " "*(max_lenght-len(full_label)+1)
print(line1)
print(line2)


EU   rejects German call to   boycott British lamb . 
B-NP B-VP    B-NP   I-NP B-VP I-VP    B-NP    I-NP O 


In [ ]:
###tokenizer
from transformers import AutoTokenizer
model_id='bert-base-cased'
tokenizer=AutoTokenizer.from_pretrained(model_id)

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

In [ ]:
inputs=tokenizer(raw_dataset['train']['words'][0],truncation=True,is_split_into_words=True)
inputs.tokens()

['[CLS]',
 'EU',
 'rejects',
 'German',
 'call',
 'to',
 'boycott',
 'British',
 'la',
 '##mb',
 '.',
 '[SEP]']

In [ ]:
inputs.word_ids()# We need to extend the labels to all tokens that share the same ID.

[None, 0, 1, 2, 3, 4, 5, 6, 7, 7, 8, None]

In [ ]:
def align_labels_with_tokens(labels,word_ids):
  current_word=None
  new_labels=[]
  for word in word_ids:
    if word!=current_word:
      current_word=word
      label=-100 if word is None else labels[word]
      new_labels.append(label)
    elif word is None:
      new_labels.append(-100)
    else:
      new_labels.append(labels[word])
  return new_labels

In [ ]:
labels=raw_dataset['train']['labels'][0]
print(labels)
print(align_labels_with_tokens(labels,inputs.word_ids()))

[11, 21, 11, 12, 21, 22, 11, 12, 0]
[-100, 11, 21, 11, 12, 21, 22, 11, 12, 12, 0, -100]


In [ ]:
### I want to perform tokenization and, at the same time, propagate the labels to all tokens that originate from the same word ID.
def tokenize_and_align_labels(examples):
  tokenize_input=tokenizer(examples['words'],truncation=True,is_split_into_words=True)
  full_labels=examples['labels']
  new_labels=[]
  for i ,label in enumerate(full_labels):
    word_id=tokenize_input.word_ids(i)
    new_labels.append(align_labels_with_tokens(label,word_id))
  tokenize_input['labels']=new_labels
  return tokenize_input

In [ ]:
tokenized_dataset=raw_dataset.map(tokenize_and_align_labels,
                                  batched=True,
                                   remove_columns=raw_dataset["train"].column_names)
tokenized_dataset

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3453
    })
})

In [ ]:
### Now let's prepared our metrics
!pip install seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=292e8b24eafcd5341e1abbf3a9836ae65cb121b7a3eb7c1f409d20b03817eba6
  Stored in directory: /root/.cache/pip/wheels/bc/92/f0/243288f899c2eacdfa8c5f9aede4c71a9bad0ee26a01dc5ead
Successfully built seqeval


In [ ]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 3.3 MB/s eta 0:00:00


In [ ]:
import evaluate
metric=evaluate.load('seqeval')

In [ ]:
### we need a padding for batch to turn our shape of input in the same shape(dynamic padding)
from transformers import DataCollatorForTokenClassification
data_collator=DataCollatorForTokenClassification(tokenizer=tokenizer)

In [ ]:
### define our compute_metrics
###We use -100 to make sure the padding tokens are ignored during loss computation
import numpy as np
def compute_metrics(eval_preds):
  logits,labels=eval_preds
  predictions=np.argmax(logits,axis=-1)
  true_labels=[[label_names[l] for l in label if l!=-100]for label in labels]
  true_predictions=[
      [ label_names[p] for (p,l) in zip(prediction,label) if l!=-100]for (prediction,label) in zip(predictions,labels)
  ]
  all_metrics = metric.compute(predictions=true_predictions, references=true_labels)
  return {
        "precision": all_metrics["overall_precision"],
        "recall": all_metrics["overall_recall"],
        "f1": all_metrics["overall_f1"],
        "accuracy": all_metrics["overall_accuracy"],
    }

In [ ]:
id2label={i:label for i,label in enumerate(label_names)}
label2id={label:i for i , label in id2label.items()}

In [ ]:
#define our model
from transformers import AutoModelForTokenClassification
model=AutoModelForTokenClassification.from_pretrained(model_id,
                                                      id2label=id2label,
                                                      label2id=label2id)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from huggingface_hub import notebook_login

notebook_login()

Token has not been saved to git credential helper.


In [ ]:
from transformers import TrainingArguments
args=TrainingArguments(

    "bert-finetuned-chunks-tags",

    save_strategy="epoch",
    learning_rate=2e-5,
    num_train_epochs=3,
    weight_decay=0.01,
    push_to_hub=True,
)

In [ ]:
from transformers import Trainer
trainer=Trainer( model=model,
    args=args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,)
trainer.train()

Step,Training Loss
500,0.108100
1000,0.091800
1500,0.079700
2000,0.067500
2500,0.051400
3000,0.049000
3500,0.053400
4000,0.032900
4500,0.040900
5000,0.038500


TrainOutput(global_step=5268, training_loss=0.06017827317854085, metrics={'train_runtime': 584.6969, 'train_samples_per_second': 72.042, 'train_steps_per_second': 9.01, 'total_flos': 920888121858078.0, 'train_loss': 0.06017827317854085, 'epoch': 3.0})

In [ ]:
trainer.push_to_hub(commit_message="Training complete")

CommitInfo(commit_url='https://huggingface.co/Alireza0017/bert-finetuned-chunks-tags/commit/13e61f950e2e775a7450f69bb85e157a227bb5f9', commit_message='Training complete', commit_description='', oid='13e61f950e2e775a7450f69bb85e157a227bb5f9', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Alireza0017/bert-finetuned-chunks-tags', endpoint='https://huggingface.co', repo_type='model', repo_id='Alireza0017/bert-finetuned-chunks-tags'), pr_revision=None, pr_num=None)

In [ ]:
from transformers import pipeline

# Replace this with your own checkpoint
model_checkpoint = "Alireza0017/bert-finetuned-chunks-tags"
token_classifier = pipeline(
    "token-classification", model=model_checkpoint, aggregation_strategy="simple"
)
token_classifier("My name is Sylvain and I work at Hugging Face in Brooklyn.")

config.json:   0%|          | 0.00/1.49k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/431M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.22k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/669k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Device set to use cuda:0


[{'entity_group': 'NP',
  'score': np.float32(0.9995522),
  'word': 'My name',
  'start': 0,
  'end': 7},
 {'entity_group': 'VP',
  'score': np.float32(0.9997267),
  'word': 'is',
  'start': 8,
  'end': 10},
 {'entity_group': 'NP',
  'score': np.float32(0.99985194),
  'word': 'S',
  'start': 11,
  'end': 12},
 {'entity_group': 'NP',
  'score': np.float32(0.9998229),
  'word': '##yl',
  'start': 12,
  'end': 14},
 {'entity_group': 'NP',
  'score': np.float32(0.9997979),
  'word': '##va',
  'start': 14,
  'end': 16},
 {'entity_group': 'NP',
  'score': np.float32(0.9998248),
  'word': '##in',
  'start': 16,
  'end': 18},
 {'entity_group': 'NP',
  'score': np.float32(0.99971825),
  'word': 'I',
  'start': 23,
  'end': 24},
 {'entity_group': 'VP',
  'score': np.float32(0.99971884),
  'word': 'work',
  'start': 25,
  'end': 29},
 {'entity_group': 'PP',
  'score': np.float32(0.9997905),
  'word': 'at',
  'start': 30,
  'end': 32},
 {'entity_group': 'NP',
  'score': np.float32(0.9998771),
  'w